# Framework for lifecycle analysis (LCA)

__author__ = "Swaminathan Sundar", "Rahul Kakodkar", "Efstratios Pistikopoulos"
__copyright__ = "Copyright 2023, Multi-parametric Optimization & Control Lab"
__credits__ = ["Swaminathan Sundar", "Rahul Kakodkar", "Efstratios N. Pistikopoulos"]
__license__ = "Open"
__version__ = "1.0.0"
__maintainer__ = "Swaminathan Sundar"
__email__ = "swaminathan.sundar@tamu.edu"
__status__ = "Production"


In [1]:
import pandas 
from numpy import poly1d, polyfit, arange
from src.energiapy.components.temporal_scale import Temporal_scale
from src.energiapy.components.resource import Resource
from src.energiapy.components.process import Process
from src.energiapy.components.material import Material
from src.energiapy.components.location import Location
from src.energiapy.components.network import Network
from src.energiapy.components.scenario import Scenario
from src.energiapy.components.transport import Transport
from src.energiapy.components.result import Result 
from src.energiapy.model.formulate import formulate, Constraints, Objective
from src.energiapy.plot import plot
from src.energiapy.model.solve import solve
import matplotlib.pyplot as plt
from matplotlib import rc
from itertools import product


**Resources**

In [2]:
SkimMilk = Resource(name = 'SkimMilk', price = 0.3, store_max = 14100*2, block = 'Input', cons_max= 14100)
PastMilk = Resource(name = 'PasteurizedMilk', price = 0, block = 'Intermediate')
Culture = Resource(name = 'Culture', price = 0,store_max =5000*2, block = 'Input', cons_max= 5000)
Whey = Resource(name = 'Whey', price = 0, block = 'Intermediate')
Curd = Resource(name = 'Curd', price = 0, store_max =6000*2, block = 'Intermediate')
SolCurd = Resource(name = 'SolCurd', price = 0, block = 'Intermediate')
Waste_1 = Resource(name = 'Waste1', price = 0, store_max =2000*2, block = 'Waste')
Cream = Resource(name = 'Cream', price = 1.5, store_max =2800*2, block = 'Input', cons_max= 2800)
Cheese = Resource(name = 'Cheese', price = 0, store_max =2000*2, block = 'Intermediate')
StdBld1 = Resource(name = 'StdBld1', revenue= 6.36, sell = True, demand = True, block = 'Output')
StdBld2 = Resource(name = 'StdBld2', revenue= 6.36, sell = True, demand = True, block = 'Output')
Protein = Resource(name = 'Protein', revenue= 0.2, store_max =3000*2,sell = True, block = 'ByProduct')
Permeate = Resource(name = 'Permeate', price = 0, sell = True, block = 'Intermediate')
Lactose = Resource(name = 'Lactose', revenue= 0.2, store_max = 3000*2, block = 'ByProduct')
Waste_2 = Resource(name = 'Waste2', revenue= 0, store_max =2000*2, block = 'Waste')
Steam = Resource(name = 'Steam', price = 0.011, block = 'Utility')
CO2 = Resource(name= 'CO2', block = 'emission', sell= True)
CH4 = Resource(name= 'CH4', block = 'emission', sell= True)
NO2 = Resource(name= 'NO2', block = 'emission', sell= True)


**Process**

In [3]:
# #Tasks
# Pasteurizer = Process(name = 'Pasteurizer', prod_min = 50, prod_max = 2700, conversion = {SkimMilk: -1, PastMilk: 1, CO2: 1.5, CH4:0.6, NO2:0.2}, \
#     cost= {'CAPEX': 0, 'Fixed O&M': 50, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 1)
# Vat1 = Process(name = 'Vat1', prod_min = 50, prod_max = 600, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5}, \
#     cost= {'CAPEX': 0, 'Fixed O&M': 75, 'Variable O&M': 0.45, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 8)
# Vat2 = Process(name = 'Vat2', prod_min = 50, prod_max = 800, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 81, 'Variable O&M': 0.5, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 8)
# Vat3 = Process(name = 'Vat3', prod_min = 50, prod_max = 1940, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 89, 'Variable O&M': 0.55, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 8)
# Drainer = Process(name = 'Drainer', prod_min = 5, prod_max = 225, conversion = {Curd: -1, SolCurd: 0.9, Waste_1: 0.1},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 1)
# Blender_1 = Process(name = 'Blender1', prod_min = 5, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 2)
# Blender_2 = Process(name = 'Blender2', prod_min = 5, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 2)
# Filler_1 = Process(name = 'Filler1', prod_min = 5, prod_max = 150, conversion = {Cheese: -1, StdBld1: 1},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 1)
# Filler_2 = Process(name = 'Filler2', prod_min = 5, prod_max = 150, conversion = {Cheese: -1, StdBld2: 1}, \
#     cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 1)
# UltraFilMem = Process(name = 'UFMem', prod_min = 50, prod_max = 5300, conversion = {Whey: -1, Protein: 0.03 , Permeate: 0.97},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 8, label = 'Ultra_Filtratiion_Membrane')
# RevOsmoMem = Process(name = 'ROMem', prod_min = 50, prod_max = 5300, conversion = {Permeate: -1, Lactose: 0.9 , Waste_2: 0.1},\
#     cost= {'CAPEX': 0, 'Fixed O&M': 120, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
#         intro_scale = 0, exit_scale= 2, label = 'Reverse_Osmosis_Membrane')


In [4]:
#Tasks
Pasteurizer = Process(name = 'Pasteurizer', prod_min = 0, prod_max = 2700, conversion = {SkimMilk: -1, PastMilk: 1, CO2: 1.5, CH4:0.6, NO2:0.2}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 50, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Vat1 = Process(name = 'Vat1', prod_min = 0, prod_max = 600, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 75, 'Variable O&M': 0.45, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Vat2 = Process(name = 'Vat2', prod_min = 0, prod_max = 800, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5},\
    cost= {'CAPEX': 0, 'Fixed O&M': 81, 'Variable O&M': 0.5, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Vat3 = Process(name = 'Vat3', prod_min = 0, prod_max = 1940, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5},\
    cost= {'CAPEX': 0, 'Fixed O&M': 89, 'Variable O&M': 0.55, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Drainer = Process(name = 'Drainer', prod_min = 0, prod_max = 225, conversion = {Curd: -1, SolCurd: 0.9, Waste_1: 0.1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Blender_1 = Process(name = 'Blender1', prod_min = 0, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2)
Blender_2 = Process(name = 'Blender2', prod_min = 0, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2)
Filler_1 = Process(name = 'Filler1', prod_min = 0, prod_max = 150, conversion = {Cheese: -1, StdBld1: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Filler_2 = Process(name = 'Filler2', prod_min = 0, prod_max = 150, conversion = {Cheese: -1, StdBld2: 1}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
UltraFilMem = Process(name = 'UFMem', prod_min = 0, prod_max = 5300, conversion = {Whey: -1, Protein: 0.03 , Permeate: 0.97},\
    cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8, label = 'Ultra_Filtratiion_Membrane')
RevOsmoMem = Process(name = 'ROMem', prod_min = 0, prod_max = 5300, conversion = {Permeate: -1, Lactose: 0.9 , Waste_2: 0.1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 120, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2, label = 'Reverse_Osmosis_Membrane')


In [5]:

Milk_Silo = Process(name = 'MilkSilo', prod_min = 0, prod_max= 2*14100,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = SkimMilk, block = 'Storage')
Culture_Silo = Process(name = 'CultureSilo', prod_min = 0, prod_max= 2*5000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Culture, block = 'Storage')
Curd_Tank_1 = Process(name = 'CurdTank1', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Curd_Tank_2 = Process(name = 'CurdTank2', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Curd_Tank_3 = Process(name = 'CurdTank3', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Waste_Tank_1 = Process(name = 'WasteTank1', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Waste_1, block = 'Storage')
Waste_Tank_2 = Process(name = 'WasteTank2', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Waste_2, block = 'Storage')
Cream_Silo = Process(name = 'CreamSilo', prod_min = 0, prod_max= 2*2800,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Cream, block = 'Storage')
Warehouse_1 = Process(name = 'Warehouse1', prod_min = 0, prod_max= 2*1000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = StdBld1, block = 'Storage')
Warehouse_2 = Process(name = 'Warehouse2', prod_min = 0, prod_max= 2*1000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = StdBld2, block = 'Storage')
Protein_Tank = Process(name = 'ProteinTank', prod_min = 0, prod_max= 2*3000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Protein, block = 'Storage')
Lactose_Tank = Process(name = 'LactoseTank', prod_min = 0, prod_max= 2*3000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Lactose, block = 'Storage')

In [6]:
scales = Temporal_scale([24])# annual working hours, each discretization represents 30 mins

**Constants**

In [7]:
processes = {Pasteurizer, Vat1, Vat2, Vat3, Drainer, Blender_1, Blender_2, UltraFilMem, \
    RevOsmoMem, Milk_Silo, Culture_Silo, Curd_Tank_1, Curd_Tank_2, Curd_Tank_3, Waste_Tank_1,\
        Waste_Tank_2, Warehouse_1, Warehouse_2, Protein_Tank, Lactose_Tank}

In [8]:
factory = Location(name= 'Factory', processes= processes, scales = scales, label= 'cheese cake factory')

In [9]:
scenario = Scenario(name= 'shell', network= factory, scales= scales, label= 'shell milp case study (HO)')

In [10]:
milp = formulate(scenario= scenario, demand = 1001, \
    constraints={Constraints.cost, Constraints.inventory, Constraints.production, Constraints.resource_balance}, \
        objective= Objective.cost)

In [11]:
# milp = formulate(scenario= scenario, \
#     constraints={Constraints.cost, Constraints.inventory, Constraints.production, Constraints.resource_balance}, \
#         objective= Objective.demand)

In [12]:
from pyomo.environ import Constraint, Set

In [13]:
def initial_inventory_rule(instance, location, resource):
    return  instance.Inv[location, resource, 0] == initial_dict[resource]

In [14]:
initial_dict = {'SkimMilk': 141, 'Cream': 28, 'Culture': 50}

In [15]:
milp.initial_inventory_cons = Constraint(milp.locations, milp.resources_purch, rule = initial_inventory_rule )
    

In [16]:
# milp.initial_inventory_cons.pprint()

In [17]:
# milp.initial_inventory_cons.pprint()
# milp.demand_constraint.pprint()

In [18]:
results = solve(scenario = scenario, instance= milp, solver= 'gurobi', \
    name=f"LCA case study", print_solversteps = True)

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (mac64[x86])
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads
Optimize a model with 7275 rows, 8228 columns and 34401 nonzeros
Model fingerprint: 0x1c20aed1
Variable types: 7296 continuous, 932 integer (932 binary)
Coefficient statistics:
  Matrix range     [3e-02, 3e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 1e+04]
Presolve removed 5286 rows and 5951 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.03 seconds (0.01 work units)
Thread count was 1 (of 4 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
    model.name="unknown";
      - termination condition: infeasible
      - message from solver: <undefined>


In [19]:
factory.__dict__.keys()

dict_keys(['name', 'processes', 'scales', 'demand', 'demand_factor', 'cost_factor', 'capacity_factor', 'demand_level', 'cost_level', 'capacity_level', 'label', 'resources', 'materials', 'scale_levels', 'resource_price', 'failure_processes', 'fail_factor'])

In [20]:
scenario.conversion.keys()

dict_keys(['Vat2', 'Pasteurizer', 'Vat3', 'ProteinTank', 'UFMem', 'Warehouse2', 'CurdTank2', 'Warehouse1', 'LactoseTank', 'Vat1', 'ROMem', 'CurdTank1', 'Blender1', 'MilkSilo', 'CurdTank3', 'Blender2', 'WasteTank2', 'WasteTank1', 'Drainer', 'CultureSilo', 'ProteinTank_discharge', 'Warehouse2_discharge', 'CurdTank2_discharge', 'Warehouse1_discharge', 'LactoseTank_discharge', 'CurdTank1_discharge', 'MilkSilo_discharge', 'CurdTank3_discharge', 'WasteTank2_discharge', 'WasteTank1_discharge', 'CultureSilo_discharge'])

In [21]:
# milp.inventory_balance_constraint.pprint()

In [22]:
milp.storage_facility_constraint.pprint()

storage_facility_constraint : storage facility sizing and location
    Size=432, Index=storage_facility_constraint_index, Active=True
    Key                                : Lower : Body                                                                      : Upper : Active
              ('Factory', 'Cheese', 0) :  -Inf :                      Cap_S[Factory,Cheese,0] - 4000*X_S[Factory,Cheese,0] :   0.0 :   True
              ('Factory', 'Cheese', 1) :  -Inf :                      Cap_S[Factory,Cheese,1] - 4000*X_S[Factory,Cheese,1] :   0.0 :   True
              ('Factory', 'Cheese', 2) :  -Inf :                      Cap_S[Factory,Cheese,2] - 4000*X_S[Factory,Cheese,2] :   0.0 :   True
              ('Factory', 'Cheese', 3) :  -Inf :                      Cap_S[Factory,Cheese,3] - 4000*X_S[Factory,Cheese,3] :   0.0 :   True
              ('Factory', 'Cheese', 4) :  -Inf :                      Cap_S[Factory,Cheese,4] - 4000*X_S[Factory,Cheese,4] :   0.0 :   True
              ('Factory', 

In [23]:
# milp.storage_facility_constraint.pprint()

In [24]:
milp.demand_constraint.pprint()

demand_constraint : specific demand for resources
    Size=48, Index=demand_constraint_index, Active=True
    Key                        : Lower  : Body                  : Upper  : Active
     ('Factory', 'StdBld1', 0) : 1001.0 :  S[Factory,StdBld1,0] : 1001.0 :   True
     ('Factory', 'StdBld1', 1) : 1001.0 :  S[Factory,StdBld1,1] : 1001.0 :   True
     ('Factory', 'StdBld1', 2) : 1001.0 :  S[Factory,StdBld1,2] : 1001.0 :   True
     ('Factory', 'StdBld1', 3) : 1001.0 :  S[Factory,StdBld1,3] : 1001.0 :   True
     ('Factory', 'StdBld1', 4) : 1001.0 :  S[Factory,StdBld1,4] : 1001.0 :   True
     ('Factory', 'StdBld1', 5) : 1001.0 :  S[Factory,StdBld1,5] : 1001.0 :   True
     ('Factory', 'StdBld1', 6) : 1001.0 :  S[Factory,StdBld1,6] : 1001.0 :   True
     ('Factory', 'StdBld1', 7) : 1001.0 :  S[Factory,StdBld1,7] : 1001.0 :   True
     ('Factory', 'StdBld1', 8) : 1001.0 :  S[Factory,StdBld1,8] : 1001.0 :   True
     ('Factory', 'StdBld1', 9) : 1001.0 :  S[Factory,StdBld1,9] : 1001.0 :